## Tratamento da Coluna estado_conservacao

In [ ]:
# Importação da biblioteca pandas, utilizada para manipulação e análise de dados tabulares
import pandas as pd

Importando a biblioteca pandas e carregando o arquivo com os dados dos imóveis.

In [ ]:
#sep=';' - separador de colunas é ponto e vírgula
#encoding='latin-1' - necessário para ler caracteres especiais do português
#low_memory=False - evita aviso de tipo misto em colunas com dados heterogêneos
df = pd.read_csv(
    '/Users/gabrielmezzalira/Documents/Faculdade/P3/BD/INTEGRACAO_BD/Planilhas/imoveis_unificado.csv',
    sep=';',
    encoding='latin-1',
    low_memory=False
)

Criando uma variável para investigar apenas a coluna desejada.

Verificamos o tipo de dado e a quantidade de entradas não nulas.

In [ ]:
#info() exibe o tipo de dado da coluna e a contagem de valores não nulos
#É esperado tipo 'object' (texto) e nenhum valor nulo
df["estado_conservacao"].info()

Verificamos a frequência de cada valor para identificar categorias esperadas e possíveis valores inválidos.

In [ ]:
#value_counts() conta a frequência de cada valor distinto na coluna
#Permite identificar categorias inesperadas ou valores inválidos misturados com os dados reais
df['estado_conservacao'].value_counts()

Obtemos um resumo estatístico da coluna, incluindo total de registros, quantidade de valores únicos e o valor mais frequente.

In [ ]:
#describe() resume a coluna: total de registros, quantidade de únicos, valor mais frequente (top) e sua contagem (freq)
df['estado_conservacao'].describe()

Calculamos o percentual de valores nulos na coluna.

In [ ]:
#Calcula o percentual de valores nulos na coluna
#isnull() retorna True/False por linha; mean() calcula a proporção; * 100 converte para percentual
#Resultado 0.0 indica ausência total de nulos
df['estado_conservacao'].isnull().mean() * 100

Listamos todos os valores distintos para verificar variações de escrita ou valores inesperados.

In [ ]:
#unique() lista todos os valores distintos presentes na coluna
#Permite verificar visualmente se há variações de escrita, valores corrompidos ou inválidos
df['estado_conservacao'].unique()

Podemos verificar que a coluna possui um valor inválido: 'estado_conservacao' aparece como dado. Isso ocorreu porque, ao unir os arquivos CSV, o cabeçalho de um dos arquivos foi incluído como linha de dados. Os demais valores, Bom, Regular e Mau, estão padronizados, com mesma capitalização e sem espaços extras.

In [ ]:
#Remove a linha onde o valor de 'estado_conservacao' é igual ao próprio nome da coluna
#Isso ocorreu porque, ao unir os CSVs incorretamente, o cabeçalho de um dos arquivos
#foi tratado como linha de dados, inserindo 'estado_conservacao' como um registro real
df = df[df['estado_conservacao'] != 'estado_conservacao']

#Confirma a distribuição após a limpeza — esperamos apenas: Bom, Regular, Mau
df['estado_conservacao'].value_counts()

Assim, removemos o valor inválido e confirmamos que apenas os três valores esperados permanecem na coluna. O tratamento dessa coluna está finalizado.

## Tratamento da Coluna logradouro

Criando uma variável para investigar apenas a coluna desejada.

Verificamos o tipo de dado e a quantidade de entradas não nulas.

In [ ]:
#info() exibe o tipo de dado e a contagem de valores não nulos
#Esperamos tipo 'object' (texto) — logradouro é uma coluna de texto livre
df["logradouro"].info()

Verificamos a frequência de cada logradouro para entender a distribuição dos dados.

In [ ]:
#Conta a frequência de cada logradouro para entender a distribuição
#Alta cardinalidade é esperada (cada rua é um valor distinto)
#Observar se há valores claramente inválidos no topo ou na cauda da lista
df['logradouro'].value_counts()

Obtemos um resumo estatístico da coluna.

In [ ]:
#describe() para colunas de texto mostra: total, únicos, valor mais frequente e sua contagem
#O campo 'unique' indica a cardinalidade real da coluna
df['logradouro'].describe()

Calculamos o percentual de valores nulos na coluna.

In [ ]:
#Calcula o percentual de nulos na coluna logradouro
#Resultado 0.0 confirma que todos os registros possuem logradouro preenchido
df['logradouro'].isnull().mean() * 100

Listamos todos os valores distintos para identificar possíveis inconsistências.

In [ ]:
#Lista todos os valores distintos para inspeção visual
#Permite identificar encoding corrompido (ex: 'josÃ©'), abreviações (ex: 'eng sampaio')
#e outros padrões inconsistentes que passariam despercebidos em análises agregadas
df['logradouro'].unique()

Utilizamos a biblioteca thefuzz para comparar logradouros com similaridade maior que 90%. Filtramos apenas os que possuem o mesmo tipo de via e tamanho parecido, evitando falsos positivos. Isso permite detectar inconsistências que passariam despercebidas em uma inspeção manual.

In [ ]:
from thefuzz import fuzz          # biblioteca de fuzzy matching (similaridade entre strings)
from itertools import combinations # gera todos os pares possíveis entre os logradouros

logradouros = df['logradouro'].unique().tolist()

suspeitos = []
for a, b in combinations(logradouros, 2):
    #Filtra: apenas compara logradouros do mesmo tipo de via (rua com rua, av com av...)
    #Isso evita falsos positivos como comparar 'rua X' com 'av Y'
    if a.split()[0] != b.split()[0]:
        continue

    #Filtra: descarta pares com diferença de tamanho maior que 30%
    #Elimina comparações como 'rua eng' vs 'rua engenheiro abreu e lima' (muito diferentes em tamanho)
    if abs(len(a) - len(b)) / max(len(a), len(b)) > 0.30:
        continue

    #token_sort_ratio ordena as palavras antes de comparar, tornando a similaridade mais robusta a diferenças de ordem entre as palavras
    score = fuzz.token_sort_ratio(a, b)

    #Apenas pares com similaridade >= 90% são considerados suspeitos de serem o mesmo logradouro
    if score >= 90:
        suspeitos.append((score, a, b))

suspeitos.sort(reverse=True)
for score, a, b in suspeitos:
    print(f"{score}% | {a} | {b}")
print(len(suspeitos))

O fuzzy matching com filtro de tamanho não captura pares onde a abreviação é muito menor que a forma completa, como 'eng' versus 'engenheiro'. Esta célula complementa a análise buscando diretamente pelos padrões conhecidos de abreviação.

In [ ]:
#O fuzzy matching com filtro de tamanho (30%) não captura pares onde a abreviação é muito menor que a forma completa (ex: 'eng' vs 'engenheiro', diferença de ~55%)
# Esta célula complementa a análise buscando diretamente pelos padrões conhecidos de abreviação

abrevs = {'eng': 'engenheiro', 'dr': 'doutor', 'gen': 'general',
          'cel': 'coronel', 'ten': 'tenente', 'prof': 'professor',
          'pe': 'padre', 'cap': 'capitao', 'gov': 'governador',
          'pref': 'prefeito', 'sen': 'senador', 'sgto': 'sargento',
          'profa': 'professora', 'maest': 'maestro'}

for abrev, completo in abrevs.items():
    #Busca logradouros que contêm a abreviação como palavra isolada (\b = word boundary)
    # regex=True → interpreta '\b' como limite de palavra, não texto literal, sem isso, '\beng\b' seria buscado literalmente e nunca encontraria nada
    com_abrev    = df[df['logradouro'].str.contains(rf'\b{abrev}\b', regex=True)]['logradouro'].unique()
    #Busca logradouros que contêm a forma completa correspondente
    com_completo = df[df['logradouro'].str.contains(rf'\b{completo}\b', regex=True)]['logradouro'].unique()

    # Só exibe se ambas as formas existem simultaneamente no dataset (inconsistência confirmada)
    if len(com_abrev) > 0 and len(com_completo) > 0:
        print(f"\n{abrev} ({len(com_abrev)}) / {completo} ({len(com_completo)}):")
        for a in com_abrev[:5]:
            print(f"  ABR:  {a}")
        for c in com_completo[:5]:
            print(f"  COMP: {c}")

Podemos verificar que a análise revelou três tipos de inconsistências na coluna logradouro. O primeiro é o encoding corrompido: logradouros com caracteres como Ã©, Ã£ e Ã§ são versões mal decodificadas de letras acentuadas, pois partes do arquivo foram gravadas em UTF-8 mas lidas como latin-1. O segundo são abreviações inconsistentes, onde o mesmo logradouro aparece com formas abreviadas e por extenso, como 'eng' e 'engenheiro', 'dr' e 'doutor', entre outros. O terceiro são pontuações inconsistentes, como apóstrofos em nomes como 'sant'anna'. Os pares identificados como ruas diferentes, por exemplo 'rua jose maria' e 'rua jose mariano', não foram tratados pois são logradouros distintos.

Assim, aplicamos as correções em sequência. Primeiro removemos a linha de cabeçalho duplicada. Em seguida corrigimos o encoding corrompido, removemos os acentos para unificar versões com e sem acento, expandimos as abreviações para a forma completa e por fim removemos pontuações inconsistentes como apóstrofos e pontos.

In [ ]:
import unicodedata

# Função para corrigir strings com encoding corrompido
# Ocorre quando caracteres UTF-8 foram gravados mas lidos como latin-1,
# gerando sequências como 'Ã©' no lugar de 'é', 'Ã£' no lugar de 'ã'
def corrigir_encoding(texto):
    try:
        return texto.encode('latin-1').decode('utf-8')
    except:
        return texto  # se não conseguir corrigir, mantém o valor original

#Função para remover acentuação de uma string
# Após corrigir o encoding, alguns logradouros ficam com acento e outros sem
#(ex: 'rua josé maria' e 'rua jose maria'). Remover acentos unifica ambos
def remover_acentos(texto):
    return unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('ascii')

#Primeiro: Remove a linha de cabeçalho duplicada
# Ao unir os CSVs, o header de um arquivo foi incluído como dado
df = df[df['logradouro'] != 'logradouro']

#Segundo: Corrige encoding corrompido (ex: 'josÃ©' → 'josé')
df['logradouro'] = df['logradouro'].apply(corrigir_encoding)

#Terceiro: Remove acentos para unificar versões com e sem acento
# Ex: 'rua josé domingues' e 'rua jose domingues' → ambos viram 'rua jose domingues'
df['logradouro'] = df['logradouro'].apply(remover_acentos)

#Quarto: Expande abreviações para a forma completa
# Lista de tuplas (padrão regex, forma completa)
# \b = word boundary, garante que só substitui a palavra isolada (não parte de outra palavra)
abrevs = [
    (r'\bgov\.', 'governador'),   # gov. com ponto (antes dos sem ponto, pois regex é guloso)
    (r'\beng\b', 'engenheiro'),
    (r'\bdr\b',  'doutor'),
    (r'\bgen\b', 'general'),
    (r'\bcel\b', 'coronel'),
    (r'\bten\b', 'tenente'),
    (r'\bprofa\b', 'professora'), # profa antes de prof para não substituir parcialmente
    (r'\bprof\b', 'professor'),
    (r'\bpe\b',  'padre'),
    (r'\bcap\b', 'capitao'),
    (r'\bgov\b', 'governador'),
    (r'\bpref\b','prefeito'),
    (r'\bsen\b', 'senador'),
    (r'\bsgto\b','sargento'),
    (r'\bmaest\b','maestro'),
]
for padrao, completo in abrevs:
    df['logradouro'] = df['logradouro'].str.replace(padrao, completo, regex=True)

# PASSO 5: Remove pontuação inconsistente e normaliza espaços múltiplos
# Ex: "sant'anna" → "santanna" | "gov." → já tratado acima | "  " → " "
df['logradouro'] = df['logradouro'].str.replace("'", '', regex=False)
df['logradouro'] = df['logradouro'].str.replace(r'\.', '', regex=True)
df['logradouro'] = df['logradouro'].str.strip()

print(f"Únicos antes do tratamento: 2237")
print(f"Únicos após tratamento: {df['logradouro'].nunique()}")
df['logradouro'].value_counts().head(10)

Com esse código, a coluna logradouro foi padronizada. O número de logradouros únicos reduziu, confirmando que as duplicatas foram unificadas. O tratamento dessa coluna está finalizado.

## Tratamento da Coluna bairro

Criando uma variável para investigar apenas a coluna desejada.

Verificamos o tipo de dado e a quantidade de entradas não nulas.

In [ ]:
# info() exibe tipo e contagem de não nulos para a coluna bairro
df['bairro'].info()

Verificamos a frequência de cada bairro para identificar possíveis valores duplicados por encoding ou espaços.

In [ ]:
# Frequência de cada bairro — permite identificar bairros com encoding corrompido
# que aparecem como entradas separadas (ex: 'Gracas' e 'GraÃ§as' são o mesmo bairro)
df['bairro'].value_counts()

Obtemos um resumo estatístico da coluna.

In [ ]:
# describe() mostra o total, a quantidade de bairros únicos e o mais frequente
# O número de únicos elevado para uma cidade indica possível fragmentação por encoding
df['bairro'].describe()

Calculamos o percentual de valores nulos na coluna.

In [ ]:
# Verifica o percentual de nulos na coluna bairro
# Resultado 0.0 indica que todos os registros possuem bairro preenchido
df['bairro'].isnull().mean() * 100

Listamos todos os valores distintos para identificar variações de escrita ou valores inesperados.

In [ ]:
# Lista todos os valores distintos para inspeção visual
# Com apenas ~95 bairros únicos, é possível identificar manualmente os problemas:
# encoding corrompido (ex: 'GraÃ§as', 'SÃ£o JosÃ©') e espaços extras ('Cohab   Ibura')
df['bairro'].unique()

Podemos verificar que a análise dos valores únicos revelou dois tipos de inconsistência. O primeiro é o encoding corrompido: os mesmos bairros aparecem em duas versões, uma sem acento e outra com caracteres corrompidos como Ã§, Ã£ e Ã. Por exemplo, 'Gracas' e 'GraÃ§as' representam o mesmo bairro, assim como 'Sao Jose' e 'SÃ£o JosÃ©', entre outros. O segundo tipo são espaços extras, como em 'Cohab   Ibura', que possui três espaços entre as palavras.

Assim, aplicamos as correções em sequência. Primeiro removemos a linha de cabeçalho duplicada. Em seguida corrigimos o encoding corrompido, removemos os acentos para unificar versões com e sem acento e por fim normalizamos os espaços extras.

In [ ]:
#Primeiro: Remove linha de cabeçalho duplicada (mesmo problema do logradouro)
df = df[df['bairro'] != 'bairro']

#Segundo: Corrige encoding corrompido usando a função definida anteriormente
# Ex: 'GraÃ§as' → 'Graças', 'SÃ£o JosÃ©' → 'São José'
df['bairro'] = df['bairro'].apply(corrigir_encoding)

#Terceiro: Remove acentos para unificar versões com e sem acento
# Ex: 'Graças' (após correção) e 'Gracas' (original sem acento) → ambos viram 'Gracas'
df['bairro'] = df['bairro'].apply(remover_acentos)

#Quarto: Normaliza espaços múltiplos
# \s+ captura um ou mais espaços consecutivos e substitui por um único espaço
# Ex: 'Cohab   Ibura' → 'Cohab Ibura'
df['bairro'] = df['bairro'].str.replace(r'\s+', ' ', regex=True).str.strip()

print(f"Únicos antes do tratamento: 95")
print(f"Únicos após tratamento: {df['bairro'].nunique()}")
df['bairro'].value_counts()

Com esse código, a coluna bairro foi padronizada. O número de bairros únicos reduziu de 95 para o valor correto, confirmando que as duplicatas foram eliminadas. O tratamento dessa coluna está finalizado.

## Validação Final

Após todos os tratamentos, verificamos o estado final das três colunas para confirmar que as correções foram aplicadas corretamente.

Comparamos o número de valores únicos antes e depois do tratamento para confirmar que as inconsistências foram eliminadas.

In [ ]:
# Compara a cardinalidade antes e depois do tratamento para cada coluna
# A redução confirma que valores que eram contados como distintos foram unificados
# (ex: 'eng sampaio' e 'engenheiro sampaio' agora são o mesmo valor)
unicos_antes = {
    'estado_conservacao': 4,   # incluía o valor inválido 'estado_conservacao'
    'logradouro':        2237,
    'bairro':              95, # incluía versões com encoding corrompido e espaços extras
}

print(f'{"Coluna":<25} {"Antes":>8} {"Depois":>8} {"Redução":>9}')
print('-' * 55)
for col, antes in unicos_antes.items():
    depois = df[col].nunique()
    print(f'{col:<25} {antes:>8} {depois:>8} {antes - depois:>9}')

Confirmamos que nenhuma das colunas tratadas possui valores nulos após o tratamento.

In [ ]:
# Confirma ausência de valores nulos após todos os tratamentos
# Esperamos 0 nulos em todas as colunas
df[['estado_conservacao', 'logradouro', 'bairro']].isnull().sum()

Exibimos a distribuição final de cada coluna para confirmar que apenas valores válidos e padronizados estão presentes.

In [ ]:
# Exibe a distribuição final de cada coluna após o tratamento completo
# estado_conservacao: deve ter apenas Bom, Regular e Mau
# bairro: sem versões duplicadas por encoding ou espaços
# logradouro: top 15 para confirmar que abreviações foram expandidas

print('=== estado_conservacao ===')
display(df['estado_conservacao'].value_counts())

print('\n=== bairro ===')
display(df['bairro'].value_counts())

print('\n=== logradouro (top 15) ===')
display(df['logradouro'].value_counts().head(15))